In [2]:
import Bio
from Bio import Align, SeqIO, Seq, SeqRecord, AlignIO, SeqFeature
from Bio.Align import Alignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# graphics imports
from reportlab.lib import colors
from reportlab.lib.units import cm
from Bio.Graphics import GenomeDiagram

# color imports
from termcolor import cprint, colored


In [3]:
WT = next(SeqIO.parse("C:\\Users\\Lyell\\Downloads\\tshz1_predicted_cDNA.fasta", "fasta")).seq
FourteenMale = next(SeqIO.parse("C:\\Users\\Lyell\\Downloads\\tshz1_predicted_14male.fasta", "fasta")).seq
fh247 = next(SeqIO.parse("C:\\Users\\Lyell\\Documents\\Reed\\THESIS\\SequenceAlignment\\tshz1_fh247.fasta", "fasta")).seq
allSeqs = next(SeqIO.parse("C:\\Users\\Lyell\\Documents\\Reed\\THESIS\\SequenceAlignment\\tshz1_all_sequences.fasta", "fasta")).seq

In [15]:
# settings for aligner
aligner = Align.PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.open_gap_score = -0.1
aligner.extend_gap_score = 0

# equalize_lengths is a helper function that adds spaces between the name of a sequence and the actual sequence
# in order to balance where they are printed on the console
def equalize_lengths (names):
    max = 0
    
    for name in names:
        if len(name) > max:
            max = len(name)

    result = []
    for name in names:
        result.append(name + ":" + " " * (max - len(name) + 1))

    return result

# return_score generates a string that represents whether all letters in a column are equal
# useful for displaying where sequences differ
def return_score (alignment, length):
    result = ""
    for x in range(length):
        if alignment[0, x].upper() == alignment[1, x].upper() == alignment[2, x].upper():
            result += "*"
        else:
            result += " "

    return result

# print_alignment takes in a sequence alignment and prints it to the console
# alignment: sequence alignment to be printed
# lineLength: desired # of letters to be printed per line
# names: list containing lables for each sequence in the alignment
# adjust: how much to offset the score string so it aligns with the sequence when printed
# features: list of tuples describing how to color each character in the sequence
# not intented to be efficient, just gets the job done
def print_alignment (alignment, lineLength, names, adjust, features):
    length = len(alignment[0])
    x = 0

    score = return_score(alignment, length)
    
    eqNames = equalize_lengths(names)
    
    while x < length:
        
        y = 0
        for seq in alignment:
            output = eqNames[y] + str(x + 1) + " "

            # decide on color for each character
            # this is very inefficient for which I am very sorry
            
            counter = x + 1
            for char in seq[x:x+lineLength]:

                # if within range of a feature, set color to it (inclusive) 
                highlight = (255, 255, 255)
                for feature in features:
                    if feature[0] <= counter and feature[1] >= counter:
                        highlight = feature[2]

                counter += 1
                output += colored(char.upper(), "black", highlight)

            print(output)
                
            y += 1

        # bad im sorry
        print(" " * (adjust + len(str(x))) + score[x:x+lineLength] + "\n")
        # print("\n")
        
        x += lineLength

# create two alignments between WT and the two mutants
alignment_new_1 = next(aligner.align(WT, FourteenMale))
alignment_new_2 = next(aligner.align(WT, fh247))

# create a combined multiple sequence alignment
combined = Alignment.from_alignments_with_same_reference([alignment_new_2, alignment_new_1])


text = colored("Aligned Genomes (WT, fh247, fourteen):\n", "black")
print(text)

# open reading frame
ORF = (192, 3575, "on_blue")

# location of SNP and induced STOP in fh247
SNPSTOP = (1518, 1520, "on_light_yellow")
SNP = (1518, 1518, "on_yellow")

# location of 5-BP deletion and induced stop in CRISPANT 14 Male
Deletion = (562, 566, "on_red")
DeletionSTOP = (593, 595, "on_light_red")

features = [ORF, SNPSTOP, SNP, Deletion, DeletionSTOP]

# adjustedAlignment is the alignment generated above (combined), but the deletion has been moved two BP to the left
# this aligns the deletion with anlysis indicating its actual position, which differed from the output of the aligner
adjustedAlignment = next(Align.parse("C:\\Users\\Lyell\\Documents\\Reed\\THESIS\\SequenceAlignment\\tshz1_adjusted_alignment.fasta", "fasta"))
print_alignment(adjustedAlignment, 150, ["WT", "fh247", "14 Male"], 10, features)

# WT protein sequence generated by translating from the start codon
WTprotein = WT[191:3575].translate(to_stop=True)
# WTprotein = SeqRecord(WTprotein, id = "WTprot")
# print(WTprotein)
# print("\n")

# fh247 protein sequence generated by translating from the start codon
fh247protein = fh247[191:3575].translate(to_stop=True)
# print(fh247protein)
# print("\n")

# CRISPANT 14 Male protein sequence generated by translating from the start codon
fourteenProtein = FourteenMale[191:3575].translate(to_stop=True)
# print(fourteenProtein)
# print("\n")

# settings to make 14 male protein align correctly
# mismatch and match scores not different
# internal deletion: negative penalty for gaps in fh247 / CRISPANT
# internal insertion: negative penalty for gaps WT
# right_deletion: positive points for large gap on the right / end of protein
# (so that in CRISPANT frameshift aligns with the rest of the protein, not at the very end)
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.extend_internal_deletion_score = -5
aligner.open_internal_deletion_score = -5
aligner.extend_internal_insertion_score = -5
aligner.open_internal_insertion_score = -5

aligner.extend_right_deletion_score = 0.1
aligner.open_right_deletion_score = 0.1

# creates alignment between WT and both mutants
fourteenProtAlign = aligner.align(WTprotein, fourteenProtein)
# print("Query: 14 male, Target: WT:\n")
# print(fourteenProtAlign[0])

fh247ProtAlign = aligner.align(WTprotein, fh247protein)
# print("Query: fh247, Target: WT:\n")
# print(fh247ProtAlign[0])

# creates and prints combined alignment
combinedProt = Alignment.from_alignments_with_same_reference([fh247ProtAlign[0], fourteenProtAlign[0]])
print("Combined prot alignment (WT, fh247, fourteen):\n")

ZF1 = (257, 281, "on_yellow")
ZF2 = (318, 342, "on_yellow")
ZF3 = (430, 454, "on_yellow")
ZF4 = (1017, 1039, "on_yellow")
ZF5 = (1084, 1107, "on_yellow")

protFeatures = [ZF1, ZF2, ZF3, ZF4, ZF5]
print_alignment(combinedProt, 150, ["WT Protein", "fh247 Protein", "14 Male Protein"], 18, protFeatures)



Aligned Genomes (WT, fh247, fourteen):

WT:      1 AGTCACATCACAAAAACCATGACAAAAAACACTGGATGACACTGATGTCAATGGTGAGCCGGCATCATATCGCAGAGTAGATTGTCAGGACTGGCTATATTTTCCATTGAGAATGCACAAAGCCATGAATTAAGCAACAGGGACATGAGG
fh247:   1 AGTCACATCACAAAAACCATGACAAAAAACACTGGATGACACTGATGTCAATGGTGAGCCGGCATCATATCGCAGAGTAGATTGTCAGGACTGGCTATATTTTCCATTGAGAATGCACAAAGCCATGAATTAAGCAACAGGGACATGAGG
14 Male: 1 AGTCACATCACAAAAACCATGACAAAAAACACTGGATGACACTGATGTCAATGGTGAGCCGGCATCATATCGCAGAGTAGATTGTCAGGACTGGCTATATTTTCCATTGAGAATGCACAAAGCCATGAATTAAGCAACAGGGACATGAGG
           ******************************************************************************************************************************************************

WT:      151 AATGAATTTGCTGAAAGAAAGACCCGGAAAACACAACTAAAATGGCCTATGTGCCGGAGGACGAGCTTAAGGCAGCTAAGATCGATGAGGAGCACCTGCAGGATGACGGACTCTCCTTAGACGGTCAGGATGCAGAGTACCTGTGCAATG
fh247:   151 AATGAATTTGCTGAAAGAAAGACCCGGAAAACACAACTAAAATGGCCTATGTGCCGGAGGACGAGCTTAAGGCAGCTAAGATCGATGAGGAGCACCTGCAGGATGACGGACTCTCCTTAGACGGTCAGGATGCA